# Hargreaves data cleaning notebook

This notebook takes raw Hargreaves measurements (two trials per paw per session) and produces a cleaned table with one row per:

- MouseID
- Sex
- Date
- Session
- Paw

and the **average withdrawal latency** for that paw-session.

It also adds a trial count column so you can spot any paw-sessions that did not have exactly 2 trials.

## 1) Imports

In [11]:
import pandas as pd
from pathlib import Path

## 2) User settings you will change when you have new data

Update these variables for each new file:

- `INPUT_PATH`: path to the new raw Excel file
- `OUTPUT_PATH`: where you want the cleaned Excel file to be written

Only change the column name variables if your raw sheet uses different headers.
Only change `SHEET_NAME` if the data are not on the first sheet or the sheet name is different.

In [12]:
# File paths (edit these)
INPUT_PATH = Path('/Users/ok28/Desktop/Analyze/Females_hargreaves_raw.xlsx')
OUTPUT_PATH = Path('/Users/ok28/Desktop/Analyze/hargreaves_females_cleaned.xlsx')

# If your data are on a specific sheet, set SHEET_NAME.
# Leave as None to read the first sheet.
SHEET_NAME = 0

# Column names (edit only if your headers differ)
COL_MOUSE = 'MouseID'
COL_SEX = 'Sex'
COL_DATE = 'Date'
COL_SESSION = 'Session'
COL_PAW = 'Paw'
COL_LATENCY = 'Latency_s'

# Grouping keys. You generally should not change this unless you add more identifiers.
GROUP_KEYS = [COL_MOUSE, COL_SEX, COL_DATE, COL_SESSION, COL_PAW]

## 3) Cleaner function

What it does:

1. Reads the raw Excel file
2. Checks required columns exist
3. Coerces latency to numeric and drops non-numeric rows
4. Groups by mouse, sex, date, session, paw
5. Computes mean latency and number of trials
6. Adds a flag if trial count is not 2
7. Writes the cleaned file

In [13]:
def clean_hargreaves(
    input_path: Path,
    output_path: Path,
    sheet_name=None,
    group_keys=None,
    col_latency: str = 'Latency_s',
):
    df = pd.read_excel(input_path, sheet_name=sheet_name)

    if group_keys is None:
        raise ValueError('group_keys must be provided')

    required = set(group_keys + [col_latency])
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}\n"
            f"Found columns: {list(df.columns)}"
        )

    # Make sure latency is numeric
    df[col_latency] = pd.to_numeric(df[col_latency], errors='coerce')
    df = df.dropna(subset=[col_latency]).copy()

    cleaned = (
        df.groupby(group_keys, as_index=False)
          .agg(
              Average_Withdrawal_Latency_s=(col_latency, 'mean'),
              N_trials=(col_latency, 'size'),
          )
    )

    cleaned['Trial_Count_Flag'] = cleaned['N_trials'].apply(
        lambda n: '' if n == 2 else f'CHECK: {n} trials'
    )

    cleaned.to_excel(output_path, index=False)
    return cleaned

## 4) Run the cleaner

This will create the cleaned Excel file at `OUTPUT_PATH` and also show the first few rows here.

In [14]:
cleaned_df = clean_hargreaves(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    sheet_name=SHEET_NAME,
    group_keys=GROUP_KEYS,
    col_latency=COL_LATENCY,
)

cleaned_df.head(10)

,MouseID,Sex,Date,Session,Paw,Average_Withdrawal_Latency_s,N_trials,Trial_Count_Flag
0,C57-1628-F0,F,2026-02-03,Baseline 1,L,17.745,2,
1,C57-1628-F0,F,2026-02-03,Baseline 1,R,13.010,2,
2,C57-1628-F0,F,2026-02-12,120 min post drug,L,6.480,2,
3,C57-1628-F0,F,2026-02-12,120 min post drug,R,9.505,2,
4,C57-1628-F0,F,2026-02-12,15 min post drug,L,1.775,2,
5,C57-1628-F0,F,2026-02-12,15 min post drug,R,9.235,2,
6,C57-1628-F0,F,2026-02-12,180 min post drug,L,5.530,2,
7,C57-1628-F0,F,2026-02-12,180 min post drug,R,9.930,2,
8,C57-1628-F0,F,2026-02-12,60 min post drug,L,8.455,2,
9,C57-1628-F0,F,2026-02-12,60 min post drug,R,12.120,2,


## 5) Quick QC: how many groups were flagged?

If anything shows up here, it means a paw-session had a trial count other than 2. The mean is still computed, but you should inspect those cases.

In [10]:
flagged = cleaned_df[cleaned_df['Trial_Count_Flag'] != '']
flagged

,MouseID,Sex,Date,Session,Paw,Average_Withdrawal_Latency_s,N_trials,Trial_Count_Flag


## 6) Optional: batch process multiple raw files

If you drop multiple `.xlsx` files into a folder, this will generate one cleaned file per raw file.

In [ ]:
# Uncomment and edit to use batch mode
# RAW_DIR = Path('raw_data')
# OUT_DIR = Path('processed_data')
# OUT_DIR.mkdir(parents=True, exist_ok=True)
#
# for f in RAW_DIR.glob('*.xlsx'):
#     out = OUT_DIR / f"{f.stem}_cleaned.xlsx"
#     _ = clean_hargreaves(
#         input_path=f,
#         output_path=out,
#         sheet_name=SHEET_NAME,
#         group_keys=GROUP_KEYS,
#         col_latency=COL_LATENCY,
#     )
#     print('Wrote:', out)